Modeling: Mass Total + Source Parametric
========================================

This script fits a multi-wavelength `Imaging` dataset of a 'galaxy-scale' strong lens with a model where:

 - The lens galaxy's light is an MGE bulge where the `ell_comps` varies across wavelength.
 - The lens galaxy's total mass distribution is an `Isothermal` and `ExternalShear`.
 - The source galaxy's light is an MGE.

Three images are fitted, corresponding to a green ('g' band), red (`r` band) and near infrared ('I' band) images.

This script assumes previous knowledge of the `multi_dataset` modeling API found in other scripts in the `multi_dataset/modeling`
package. If anything is unclear check those scripts out.

__Contents__

- **Effective Radius vs Wavelength:** Unlike other `multi_dataset` modeling scripts, the effective radius of the lens and source galaxies as a.
- **Colors:** The colors of the multi-wavelength image, which in this case are green (g-band) and red (r-band).
- **Wavelengths:** The effective_radius of each source galaxy is parameterized as a function of wavelength.
- **Pixel Scales:** Every multi-wavelength dataset can have its own unique pixel-scale.
- **Dataset & Mask:** Standard set up of the dataset and mask that is fitted.
- **Analysis:** Create the Analysis object that defines how the model is fitted to the data.
- **Model:** Compose the lens model fitted to the data.
- **Shared Source Mesh (Pixelization):** Reconstruct every band's source on one shared Delaunay mesh via `shared_preloads=True`.
- **Search:** Configure the non-linear search used to fit the model.
- **Result:** Overview of the results of the model-fit.

__Effective Radius vs Wavelength__

Unlike other `multi_dataset` modeling scripts, the effective radius of the lens and source galaxies as a user defined function of
wavelength, for example following a relation `y = (m * x) + c` -> `effective_radius = (m * wavelength) + c`.

By using a linear relation `y = mx + c` the free parameters are `m` and `c`, which does not scale with the number
of datasets. For datasets with multi-wavelength images (e.g. 5 or more) this allows us to parameterize the variation
of parameters across the datasets in a way that does not lead to a very complex parameter space.

For example, in other scripts, a free `effective_radius` is created for every datasets, which would add 5+ free parameters
to the model for 5+ datasets.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autolens import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autolens")

In [ ]:

from autolens import jax_wrapper  # Sets JAX environment before other imports

# from autolens import setup_notebook; setup_notebook()

from pathlib import Path

import autofit as af
import autolens as al
import autolens.plot as aplt

__Colors__

The colors of the multi-wavelength image, which in this case are green (g-band) and red (r-band).

The strings are used for load each dataset.

In [ ]:
waveband_list = ["g", "r"]  # , "I"]

__Wavelengths__

The effective_radius of each source galaxy is parameterized as a function of wavelength.

Therefore we define a list of wavelengths of each color above.

In [ ]:
wavelength_list = [464, 658, 806]

__Pixel Scales__

Every multi-wavelength dataset can have its own unique pixel-scale.

In [ ]:
pixel_scales_list = [0.08, 0.12, 0.12]

__Dataset__

Load and plot each multi-wavelength strong lens dataset, using a list of their waveband colors.

In [ ]:
dataset_type = "multi_dataset"
dataset_label = "imaging"
dataset_name = "wavelength_dependence"

dataset_path = Path("dataset") / dataset_type / dataset_label / dataset_name

If the dataset does not already exist on your system, it is created by running the corresponding simulator
script. This ensures the example can be run without manually simulating the data first.

In [ ]:
if al.util.dataset.should_simulate(str(dataset_path)):
    import subprocess
    import sys

    subprocess.run(
        [sys.executable, "scripts/multi_dataset/features/wavelength_dependence/simulator.py"],
        check=True,
    )

dataset_list = [
    al.Imaging.from_fits(
        data_path=Path(dataset_path) / f"{waveband}_data.fits",
        psf_path=Path(dataset_path) / f"{waveband}_psf.fits",
        noise_map_path=Path(dataset_path) / f"{waveband}_noise_map.fits",
        pixel_scales=pixel_scales,
    )
    for waveband, pixel_scales in zip(waveband_list, pixel_scales_list)
]

for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Mask__

Define a 3.0" circular mask, which includes the emission of the lens and source galaxies.

For multi-wavelength lens modeling, we use the same mask for every dataset whenever possible. This is not
absolutely necessary, but provides a more reliable analysis.

In [ ]:
mask_list = [
    al.Mask2D.circular(
        shape_native=dataset.shape_native, pixel_scales=dataset.pixel_scales, radius=3.0
    )
    for dataset in dataset_list
]

dataset_list = [
    dataset.apply_mask(mask=mask) for imaging, mask in zip(dataset_list, mask_list)
]

for dataset in dataset_list:
    aplt.subplot_imaging_dataset(dataset=dataset)

__Analysis__

We create an `Analysis` object for every dataset.

In [ ]:
analysis_list = [al.AnalysisImaging(dataset=dataset) for dataset in dataset_list]

__Model__

We compose a lens model where:

 - The lens galaxy's total mass distribution is an `Isothermal` and `ExternalShear` [7 parameters].

 - The source galaxy's light is an MGE with 1 x 20 Gaussians [4 parameters].

The number of free parameters and therefore the dimensionality of non-linear parameter space is N=15.

In [ ]:
lens = af.Model(
    al.Galaxy,
    redshift=0.5,
    bulge=al.lp_linear.Sersic,
    mass=al.mp.Isothermal,
    shear=al.mp.ExternalShear,
)

source = af.Model(al.Galaxy, redshift=1.0, bulge=al.lp_linear.SersicCore)

model = af.Collection(galaxies=af.Collection(lens=lens, source=source))

__Model + Analysis__

We now make the lens and source `effective_radius` a free parameter across every analysis object.

Unlike other scripts, where the `effective_radius` for every dataset is created as a free parameter, we will assume that 
the `effective_radius` of the lens and source galaxies linearly varies as a function of wavelength, and therefore compute 
the `effective_radius` value for each color image using a linear relation `y = mx + c`.

The function below is not used to compose the model, but illustrates how the `effective_radius` values were computed
in the corresponding `wavelength_dependence` simulator script.

In [ ]:


def lens_effective_radius_from(wavelength):
    m = 1.0 / 100.0  # lens appears brighter with wavelength
    c = 3

    return m * wavelength + c


def source_effective_radius_from(wavelength):
    m = -(1.2 / 100.0)  # source appears fainter with wavelength
    c = 10

    return m * wavelength + c


To parameterize the above relation as a model, we compose `m` and `c` as priors and use PyAutoFit's prior arithmatic
to compose a model as a linear relation.

In [ ]:
lens_m = af.UniformPrior(lower_limit=-0.1, upper_limit=0.1)
lens_c = af.UniformPrior(lower_limit=-10.0, upper_limit=10.0)

source_m = af.UniformPrior(lower_limit=-0.1, upper_limit=0.1)
source_c = af.UniformPrior(lower_limit=-10.0, upper_limit=10.0)

The free parameters of our model there are no longer `effective_radius` values, but the parameters `m` and `c` in the relation
above. 

The model complexity therefore does not increase as we add more parameters to the model.

In [ ]:

analysis_factor_list = []

for wavelength, analysis in zip(wavelength_list, analysis_list):
    lens_effective_radius = (wavelength * lens_m) + lens_c
    source_effective_radius = (wavelength * source_m) + source_c

    model_analysis = model.copy()

    model_analysis.galaxies.lens.bulge.effective_radius = lens_effective_radius
    model_analysis.galaxies.source.bulge.effective_radius = source_effective_radius

    analysis_factor = af.AnalysisFactor(prior_model=model_analysis, analysis=analysis)

    analysis_factor_list.append(analysis_factor)

The factor graph is created and its info can be printed after the relational model has been defined.

In [ ]:
factor_graph = af.FactorGraphModel(*analysis_factor_list, use_jax=True)

print(factor_graph.global_prior_model.info)

__Shared Source Mesh (Pixelization)__

When the source is reconstructed on a pixelization, every band can reconstruct its source on the identical
shared Delaunay mesh by setting `shared_preloads=True` on each `AnalysisImaging` (see
`features/same_wavelength/modeling.py` for the full description):

    analysis_list = [
        al.AnalysisImaging(dataset=dataset, adapt_images=adapt_images, shared_preloads=True)
        for dataset in dataset_list
    ]

Because the lens model is shared, the source-plane mesh is band-invariant and is ray-traced once by the lead
factor. Each band still solves for its own reconstruction against its own data — exactly what multi-wavelength
modeling needs, since the source's appearance varies with wavelength (its colour). The shared mesh makes the
per-band reconstructions directly comparable pixel-by-pixel, which is what turns them into a resolved colour
map of the source.

__Search__

The model is fitted to the data using the nested sampling algorithm Nautilus (see `start.here.py` for a 
full description).

In [ ]:
search = af.Nautilus(
    path_prefix=Path("multi_dataset", "modeling"),
    name="wavelength_dependence",
    unique_tag=dataset_name,
    n_live=100,
    n_batch=50,  # GPU lens model fits are batched and run simultaneously, see VRAM section below.
    live_visual_update=False,  # Set True to open a live matplotlib window (script) or refresh a Jupyter cell (notebook).
)

__Model-Fit__

In [ ]:
result_list = search.fit(model=factor_graph.global_prior_model, analysis=factor_graph)

__Result__

The result object returned by this model-fit is a list of `Result` objects, because we used a factor graph.
Each result corresponds to each analysis, and therefore corresponds to the model-fit at that wavelength.

For example, close inspection of the `max_log_likelihood_instance` of the two results shows that all parameters,
except the `effective_radius` of the source galaxy's `bulge`, are identical.

In [ ]:
print(result_list[0].max_log_likelihood_instance)
print(result_list[1].max_log_likelihood_instance)

Plotting each result's tracer shows that the source appears different, owning to its different intensities.

In [ ]:
for result in result_list:
    aplt.subplot_tracer(tracer=result.max_log_likelihood_tracer, grid=result.grids.lp)

    aplt.subplot_fit_imaging(fit=result.max_log_likelihood_fit)

The `Samples` object still has the dimensions of the overall non-linear search (in this case N=15). 

Therefore, the samples is identical in every result object.

In [ ]:
for result in result_list:
    aplt.corner_anesthetic(samples=result.samples)

Checkout `autolens_workspace/*/guides/results` for a full description of analysing results.